In [ ]:
# 随机重命名文件（001–999）
import os
import random

# 指定目标文件夹（修改为你的路径）
target_folder = r"G:\1002Video\1920x1080"

# 获取文件列表（只当前文件夹）
files = [f for f in os.listdir(target_folder) if os.path.isfile(os.path.join(target_folder, f))]

# 随机编号池（001 ~ 999）
available_numbers = list(range(1, 1000))
random.shuffle(available_numbers)

for i, filename in enumerate(files):
    old_path = os.path.join(target_folder, filename)
    name, ext = os.path.splitext(filename)

    if not available_numbers:
        print("编号用完！最多支持 999 个文件。")
        break

    number = available_numbers.pop()
    new_name = f"{number:03d}{ext}"
    new_path = os.path.join(target_folder, new_name)

    # 如果新文件名已存在，重复取号（保证不冲突）
    while os.path.exists(new_path):
        if not available_numbers:
            print("编号用完！最多支持 999 个文件。")
            break
        number = available_numbers.pop()
        new_name = f"{number:03d}{ext}"
        new_path = os.path.join(target_folder, new_name)

    os.rename(old_path, new_path)
    print(f"{filename} → {new_name}")

print("重命名完成！")


In [1]:
import subprocess
import shutil

# 确定 yt-dlp 路径
yt_dlp_path = shutil.which("yt-dlp") or "C:/完整路径/yt-dlp.exe"

cmd = [
    yt_dlp_path,
    "https://www.youtube.com/watch?v=T8AtgqtUZ80",
    "--print", "filename",
    "--cookies", "C:\\02Programmer\\02Proj\PyVSCode\ConfigPrivate\Rep001Tools002YTProjV3\cookies.txt"
]

result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='ignore')

print("Return code:", result.returncode)
print("STDOUT:", result.stdout.strip())
print("STDERR:", result.stderr.strip())


Return code: 0
STDOUT: get WEAR with me) office work outfit   [T8AtgqtUZ80].webm
STDERR: 


In [ ]:
import os
import subprocess

# 设置你的目标文件夹
folder = r""

# 支持的视频扩展名
video_extensions = [".mp4", ".mov", ".mkv", ".avi", ".webm"]

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    name, ext = os.path.splitext(filename)
    if ext.lower() in video_extensions:
        output_file = os.path.join(folder, f"{name}_rotated{ext}")
        cmd = [
            "ffmpeg", "-i", file_path,
            "-vf", "transpose=2",  # 逆时针 90°
            "-c:a", "copy",        # 音频流直接复制
            # "-c:v", "copy",        # 视频流直接复制（只修改 metadata）
            # "-metadata:s:v", "rotate=270",  # 设置 metadata // 快但是要播放器支持
            output_file, "-y"
        ]
        print(f"正在处理: {filename}")
        subprocess.run(cmd)

print("✅ 所有文件处理完成！")


In [ ]:
import telethon

In [ ]:
import os
import subprocess
import tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed

# 输入文件夹
root_folder = r""

# 输出文件夹
output_folder = root_folder + "_merged"
os.makedirs(output_folder, exist_ok=True)

# 每组目标大小（25MB）
target_group_size = 25 * 1024 * 1024

# 输出比特率
output_bitrate = "320k"

# 支持的音频格式
audio_extensions = [".mp3", ".m4a", ".flac", ".wav", ".ogg", ".aac", ".opus"]

# 最大线程数
max_workers = 4

def build_output_name(subdir, group_number, file_paths):
    base_names = [os.path.splitext(os.path.basename(f))[0] for f in file_paths[:5]]
    name_part = "_".join(base_names)
    if len(name_part) > 150:
        name_part = name_part[:150]
    subdir_name = os.path.basename(subdir)
    return f"{subdir_name}_group{group_number:03d}_{name_part}.mp3"

def process_subdir(subdir, files):
    audio_files = []
    for file in files:
        file_path = os.path.join(subdir, file)
        ext = os.path.splitext(file)[1].lower()
        if ext in audio_extensions:
            size = os.path.getsize(file_path)
            audio_files.append((size, file_path))

    if not audio_files:
        print(f"跳过（没有音频文件）：{subdir}")
        return

    audio_files.sort()
    group = []
    group_size = 0
    group_number = 1

    for size, file_path in audio_files:
        group.append(file_path)
        group_size += size

        if group_size >= target_group_size:
            handle_group(subdir, group, group_number)
            group = []
            group_size = 0
            group_number += 1

    # 最后一组（不足25MB）
    if group:
        handle_group(subdir, group, group_number)

def handle_group(subdir, group, group_number):
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_mp3_files = []
        for i, src_file in enumerate(group):
            tmp_mp3 = os.path.join(tmpdir, f"temp_{i:03d}.mp3")
            cmd_convert = [
                "ffmpeg", "-i", src_file, "-c:a", "libmp3lame", "-b:a", output_bitrate, tmp_mp3, "-y", "-loglevel", "error"
            ]
            result = subprocess.run(cmd_convert, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            if result.returncode == 0:
                tmp_mp3_files.append(tmp_mp3)
            else:
                print(f"❌ 转码失败: {src_file}")
                print(result.stderr)

        if not tmp_mp3_files:
            print(f"⚠️ 没有可用文件合并: {subdir}")
            return

        list_file = os.path.join(tmpdir, "file_list.txt")
        with open(list_file, "w", encoding="utf-8") as f:
            for mp3 in tmp_mp3_files:
                fixed_mp3 = mp3.replace('\\', '/')
                f.write(f"file '{fixed_mp3}'\n")

        output_file = os.path.join(output_folder, build_output_name(subdir, group_number, group))
        cmd_concat = [
            "ffmpeg", "-f", "concat", "-safe", "0",
            "-i", list_file, "-c:a", "libmp3lame", "-b:a", output_bitrate, output_file, "-y", "-loglevel", "error"
        ]
        result = subprocess.run(cmd_concat, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        if result.returncode == 0:
            print(f"✅ 合并完成: {output_file}")
        else:
            print(f"❌ 合并失败: {output_file}")
            print(result.stderr)

def main():
    tasks = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for subdir, dirs, files in os.walk(root_folder):
            if files:
                tasks.append(executor.submit(process_subdir, subdir, files))

        for future in as_completed(tasks):
            try:
                future.result()
            except Exception as e:
                print(f"⚠️ 子任务异常: {e}")

    print("🎉 全部子文件夹处理完成！")
 
if __name__ == "__main__":
    main()


In [ ]:
# 拆散文件夹
import os
import shutil

# 主文件夹路径
root_folder = r""

def move_files_to_root_folder(root_folder):
    # 遍历根目录下的所有文件夹
    for subdir, dirs, files in os.walk(root_folder, topdown=False):
        # 跳过根文件夹本身
        if subdir == root_folder:
            continue
        
        for file in files:
            # 构造源文件路径
            source_file = os.path.join(subdir, file)
            # 构造目标文件路径
            destination_file = os.path.join(root_folder, file)
            
            # 如果目标路径已经有同名文件，进行重命名
            if os.path.exists(destination_file):
                name, ext = os.path.splitext(file)
                counter = 1
                while os.path.exists(destination_file):
                    destination_file = os.path.join(root_folder, f"{name}_{counter}{ext}")
                    counter += 1
            
            # 移动文件到根文件夹
            shutil.move(source_file, destination_file)
            print(f"文件已移动: {source_file} -> {destination_file}")
        
        # 删除空的子文件夹
        os.rmdir(subdir)
        print(f"已删除空文件夹: {subdir}")

# 调用函数
move_files_to_root_folder(root_folder)
